<a href="https://colab.research.google.com/github/Ashwini9713/threat-intelligence/blob/main/exp16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from datetime import datetime, timedelta

with open("access.log", "w") as f:
    start = datetime(2026, 7, 16, 13, 45, 0)

    # Attacker IP - 15 requests in 10 seconds
    for i in range(15):
        ts = start + timedelta(seconds=i)
        f.write(f'192.168.1.10 [{ts.strftime("%Y-%m-%d %H:%M:%S")}] "GET /index.html" 200\n')

    # Normal IP - 3 requests spread out
    f.write(f'10.0.0.5 [2026-07-16 13:46:00] "GET /about.html" 200\n')
    f.write(f'10.0.0.5 [2026-07-16 13:46:30] "GET /home.html" 200\n')
    f.write(f'10.0.0.5 [2026-07-16 13:47:00] "GET /contact.html" 200\n')

print("Log file created.")

Log file created.


In [7]:
from collections import defaultdict
from datetime import datetime

# ---- SETTINGS ----
REQUEST_LIMIT = 10          # max requests allowed
TIME_WINDOW = 30            # seconds
LOG_FILE = "access.log"

def read_log(file_path):
    entries = []
    with open(file_path, "r") as f:
        for line in f:
            parts = line.strip().split(" ", 2)
            if len(parts) < 2:
                continue
            ip = parts[0]
            time_str = parts[1].strip("[]")
            try:
                ts = datetime.strptime(time_str, "%Y-%m-%d %H:%M:%S")
                entries.append((ip, ts))
            except ValueError:
                continue
    return entries

def detect_dos(entries):
    ip_requests = defaultdict(list)

    for ip, ts in entries:
        ip_requests[ip].append(ts)

    print("---- Scan Results ----\n")
    dos_found = False

    for ip, timestamps in ip_requests.items():
        timestamps.sort()
        count = len(timestamps)
        duration = (timestamps[-1] - timestamps[0]).total_seconds()

        print(f"IP: {ip} -> {count} requests")

        if count >= REQUEST_LIMIT and duration <= TIME_WINDOW:
            print(f"  [ALERT] Possible DoS attack! {count} requests in {duration:.1f} seconds\n")
            dos_found = True
        else:
            print(f"  Normal traffic (spread over {duration:.1f} seconds)\n")

    if not dos_found:
        print("No DoS attack detected.")


# ---- MAIN ----
entries = read_log(LOG_FILE)
detect_dos(entries)

---- Scan Results ----

No DoS attack detected.
